# BBD–RSM and fully nested condition-level ML reproducibility

This notebook contains the final analysis workflow supporting the revised manuscript.

**Important:** exploratory run-level, frozen-hyperparameter, and repeated-inner screening analyses used during model development are intentionally omitted. The reported ML performance is based only on the fully nested condition-level LOOCV implemented below.


In [ ]:
# ============================================================
# FINAL RSM REANALYSIS
# Based on the confirmed final experimental datasets
#
# Experimental structure:
#   15 BBD runs
#   × 2 independent saccharification experiments per run
#   = 30 independent response observations
#
# A = Enzyme loading (FPU/g-glucan)
# B = Substrate concentration (%, w/v)
# C = KMF006 blending ratio (%, v/v)
#
# Coded variables:
#   A = (FPU - 20) / 10
#   B = (Substrate - 11) / 2
#   C = (KMF006 ratio - 30) / 20
# ============================================================

import re
import numpy as np
import pandas as pd

from scipy import stats
from scipy.optimize import differential_evolution


# ============================================================
# 1. FILE PATHS
# ============================================================

WITHOUT_BARK_FILE = "data/raw_without_bark.xlsx"
WITH_BARK_FILE    = "data/raw_with_bark.xlsx"


# ============================================================
# 2. SAVE OPTION
# ============================================================

SAVE_RESULTS = False


# ============================================================
# 3. READ FINAL DATASET
# ============================================================

def read_final_dataset(file_path, suffix):

    df = pd.read_excel(file_path)

    print("\nOriginal columns:")
    print(df.columns.tolist())

    # --------------------------------------------------------
    # Rename columns
    # --------------------------------------------------------

    first_col = df.columns[0]

    # Saccharification column name can differ slightly
    sac_cols = [
        c for c in df.columns
        if str(c).lower().startswith("saccharificationrate")
    ]

    if len(sac_cols) != 1:
        raise ValueError(
            f"Could not uniquely identify saccharification column.\n"
            f"Candidates: {sac_cols}"
        )

    sac_col = sac_cols[0]

    df = df.rename(
        columns={
            first_col: "Experimental_Run",
            "Enzymeratio": "Ratio",
            sac_col: "Saccharification"
        }
    )

    required = [
        "Experimental_Run",
        "FPU",
        "Substrate",
        "Ratio",
        "Saccharification"
    ]

    missing = [
        c for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing columns: {missing}"
        )

    df = df[required].copy()

    # --------------------------------------------------------
    # Clean run ID
    # --------------------------------------------------------

    df["Experimental_Run"] = (
        df["Experimental_Run"]
        .astype(str)
        .str.strip()
    )

    # Numeric run number
    df["Run_number"] = pd.to_numeric(
        df["Experimental_Run"]
        .str.extract(r"(\d+)", expand=False),
        errors="coerce"
    )

    # Expected suffix check
    bad_suffix = ~df["Experimental_Run"].str.endswith(suffix)

    if bad_suffix.any():
        print(
            "\nWARNING: unexpected run labels:"
        )
        print(
            df.loc[
                bad_suffix,
                "Experimental_Run"
            ].unique()
        )

    # --------------------------------------------------------
    # Numeric conversion
    # --------------------------------------------------------

    for col in [
        "FPU",
        "Substrate",
        "Ratio",
        "Saccharification"
    ]:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    df = df.dropna(
        subset=[
            "Run_number",
            "FPU",
            "Substrate",
            "Ratio",
            "Saccharification"
        ]
    ).copy()

    return df


# ============================================================
# 4. EXTRACT BBD RUNS 1–15
# ============================================================

def extract_bbd(df, condition_name):

    bbd = (
        df[
            (df["Run_number"] >= 1) &
            (df["Run_number"] <= 15)
        ]
        .copy()
        .sort_values(
            [
                "Run_number",
                "Experimental_Run"
            ]
        )
        .reset_index(drop=True)
    )

    print("\n")
    print("=" * 100)
    print(f"BBD AUDIT — {condition_name.upper()}")
    print("=" * 100)

    print(
        "BBD observations:",
        len(bbd)
    )

    print(
        "BBD run IDs:",
        bbd["Experimental_Run"].nunique()
    )

    # --------------------------------------------------------
    # Replicate number per BBD run
    # --------------------------------------------------------

    counts = (
        bbd
        .groupby("Experimental_Run")
        .size()
        .sort_index()
    )

    print("\nObservations per BBD run:")
    print(
        counts.to_string()
    )

    if len(bbd) != 30:
        raise ValueError(
            f"{condition_name}: expected 30 BBD observations, "
            f"but found {len(bbd)}."
        )

    if bbd["Experimental_Run"].nunique() != 15:
        raise ValueError(
            f"{condition_name}: expected 15 BBD runs."
        )

    if not (counts == 2).all():
        raise ValueError(
            f"{condition_name}: not every BBD run contains "
            f"exactly two experimental replicates."
        )

    # --------------------------------------------------------
    # Check that factors are constant inside a BBD run
    # --------------------------------------------------------

    factor_check = (
        bbd
        .groupby("Experimental_Run")
        [["FPU", "Substrate", "Ratio"]]
        .nunique()
    )

    if (factor_check > 1).any().any():
        raise ValueError(
            "Factor settings differ within one BBD run."
        )

    return bbd


# ============================================================
# 5. TABLE 1 SUMMARY
# ============================================================

def make_table1_summary(bbd):

    summary = (
        bbd
        .groupby(
            [
                "Run_number",
                "Experimental_Run",
                "FPU",
                "Substrate",
                "Ratio"
            ],
            as_index=False
        )
        .agg(
            Mean_SR=(
                "Saccharification",
                "mean"
            ),

            SD_SR=(
                "Saccharification",
                "std"
            ),

            Replicate_1=(
                "Saccharification",
                "first"
            ),

            Replicate_2=(
                "Saccharification",
                "last"
            ),

            n=(
                "Saccharification",
                "size"
            )
        )
        .sort_values(
            "Run_number"
        )
        .reset_index(drop=True)
    )

    return summary


# ============================================================
# 6. CODE FACTORS
# ============================================================

def add_coded_variables(df):

    d = df.copy()

    d["A"] = (
        d["FPU"] - 20.0
    ) / 10.0

    d["B"] = (
        d["Substrate"] - 11.0
    ) / 2.0

    d["C"] = (
        d["Ratio"] - 30.0
    ) / 20.0

    return d


# ============================================================
# 7. DESIGN MATRIX
# ============================================================

def build_design_matrix(df):

    A = df["A"].to_numpy(dtype=float)
    B = df["B"].to_numpy(dtype=float)
    C = df["C"].to_numpy(dtype=float)

    X = np.column_stack(
        [
            np.ones(len(df)),
            A,
            B,
            C,
            A**2,
            B**2,
            C**2,
            A * B,
            A * C,
            B * C
        ]
    )

    names = [
        "Intercept",
        "A",
        "B",
        "C",
        "A²",
        "B²",
        "C²",
        "AB",
        "AC",
        "BC"
    ]

    return X, names


# ============================================================
# 8. FIT FULL QUADRATIC RSM
# ============================================================

def fit_rsm(bbd, condition_name):

    d = add_coded_variables(
        bbd
    )

    X, names = build_design_matrix(
        d
    )

    y = d[
        "Saccharification"
    ].to_numpy(dtype=float)

    n = len(y)
    p = X.shape[1]

    # --------------------------------------------------------
    # OLS coefficients
    # --------------------------------------------------------

    beta = np.linalg.lstsq(
        X,
        y,
        rcond=None
    )[0]

    y_pred = X @ beta

    residual = y - y_pred

    # --------------------------------------------------------
    # Overall sums of squares
    # --------------------------------------------------------

    y_mean = np.mean(y)

    SS_total = np.sum(
        (y - y_mean)**2
    )

    SS_residual = np.sum(
        residual**2
    )

    SS_model = (
        SS_total -
        SS_residual
    )

    df_model = p - 1
    df_residual = n - p
    df_total = n - 1

    MS_model = (
        SS_model /
        df_model
    )

    MS_residual = (
        SS_residual /
        df_residual
    )

    F_model = (
        MS_model /
        MS_residual
    )

    p_model = stats.f.sf(
        F_model,
        df_model,
        df_residual
    )

    # --------------------------------------------------------
    # R²
    # --------------------------------------------------------

    R2 = (
        1 -
        SS_residual /
        SS_total
    )

    Adj_R2 = (
        1 -
        (1 - R2) *
        (n - 1) /
        (n - p)
    )

    # --------------------------------------------------------
    # Coefficient uncertainty
    # --------------------------------------------------------

    XtX_inv = np.linalg.inv(
        X.T @ X
    )

    SE = np.sqrt(
        np.diag(XtX_inv) *
        MS_residual
    )

    t_value = (
        beta /
        SE
    )

    p_value = (
        2 *
        stats.t.sf(
            np.abs(t_value),
            df_residual
        )
    )

    # --------------------------------------------------------
    # Partial SS / F for each individual term
    #
    # For each 1-df coefficient:
    # F = t²
    # SS_partial = F × MSE
    # --------------------------------------------------------

    term_F = (
        t_value[1:]**2
    )

    term_SS = (
        term_F *
        MS_residual
    )

    term_p = (
        stats.f.sf(
            term_F,
            1,
            df_residual
        )
    )

    coefficient_table = pd.DataFrame(
        {
            "Term": names,

            "Coefficient":
                beta,

            "SE":
                SE,

            "t":
                t_value,

            "p":
                p_value
        }
    )

    term_table = pd.DataFrame(
        {
            "Term":
                names[1:],

            "DF":
                1,

            "SS":
                term_SS,

            "MS":
                term_SS,

            "Coefficient":
                beta[1:],

            "F":
                term_F,

            "p":
                term_p
        }
    )

    # ========================================================
    # 9. PURE ERROR AND LACK OF FIT
    # ========================================================
    #
    # Pure error is calculated from repeated observations
    # obtained at identical experimental factor settings.
    #
    # There are 13 unique factor combinations in a
    # conventional 3-factor BBD:
    #
    #   12 edge points
    #   + 1 center-point setting
    #
    # Center-point runs remain separate experimental
    # observations, but all share the same X coordinates.
    # ========================================================

    group_cols = [
        "FPU",
        "Substrate",
        "Ratio"
    ]

    unique_design_points = (
        d[group_cols]
        .drop_duplicates()
        .shape[0]
    )

    SS_pure_error = 0.0
    df_pure_error = 0

    for _, group in d.groupby(
        group_cols
    ):

        yi = group[
            "Saccharification"
        ].to_numpy(
            dtype=float
        )

        group_mean = np.mean(
            yi
        )

        SS_pure_error += np.sum(
            (yi - group_mean)**2
        )

        df_pure_error += (
            len(yi) - 1
        )

    # --------------------------------------------------------
    # Lack-of-fit
    # --------------------------------------------------------

    SS_lof = (
        SS_residual -
        SS_pure_error
    )

    df_lof = (
        df_residual -
        df_pure_error
    )

    MS_pure_error = (
        SS_pure_error /
        df_pure_error
    )

    MS_lof = (
        SS_lof /
        df_lof
    )

    F_lof = (
        MS_lof /
        MS_pure_error
    )

    p_lof = stats.f.sf(
        F_lof,
        df_lof,
        df_pure_error
    )

    # ========================================================
    # 10. ANOVA SUMMARY
    # ========================================================

    anova = pd.DataFrame(
        {
            "Source": [
                "Model",
                "Lack of fit",
                "Pure error",
                "Residual",
                "Total"
            ],

            "DF": [
                df_model,
                df_lof,
                df_pure_error,
                df_residual,
                df_total
            ],

            "SS": [
                SS_model,
                SS_lof,
                SS_pure_error,
                SS_residual,
                SS_total
            ],

            "MS": [
                MS_model,
                MS_lof,
                MS_pure_error,
                MS_residual,
                np.nan
            ],

            "F": [
                F_model,
                F_lof,
                np.nan,
                np.nan,
                np.nan
            ],

            "p": [
                p_model,
                p_lof,
                np.nan,
                np.nan,
                np.nan
            ]
        }
    )

    # ========================================================
    # 11. CONSTRAINED OPTIMIZATION
    # ========================================================
    #
    # IMPORTANT:
    # Search only inside the ORIGINAL BBD domain:
    #
    #   A: 10–30 FPU/g-glucan
    #   B: 9–13%
    #   C: 10–50%
    #
    # coded domain = [-1, +1] for A, B, C
    # ========================================================

    def predict_coded(x):

        A, B, C = x

        row = np.array(
            [
                1,
                A,
                B,
                C,
                A**2,
                B**2,
                C**2,
                A * B,
                A * C,
                B * C
            ],
            dtype=float
        )

        return float(
            row @ beta
        )


    optimization = differential_evolution(
        func=lambda x:
            -predict_coded(x),

        bounds=[
            (-1, 1),
            (-1, 1),
            (-1, 1)
        ],

        seed=2026,

        polish=True,

        tol=1e-12
    )


    A_opt, B_opt, C_opt = (
        optimization.x
    )

    predicted_optimum = (
        -optimization.fun
    )


    FPU_opt = (
        20 +
        10 * A_opt
    )

    Substrate_opt = (
        11 +
        2 * B_opt
    )

    Ratio_opt = (
        30 +
        20 * C_opt
    )


    optimum = pd.DataFrame(
        {
            "Condition": [
                condition_name
            ],

            "FPU": [
                FPU_opt
            ],

            "Substrate": [
                Substrate_opt
            ],

            "KMF006_ratio": [
                Ratio_opt
            ],

            "Predicted_SR": [
                predicted_optimum
            ]
        }
    )


    # ========================================================
    # 12. PRINT RESULTS
    # ========================================================

    print("\n")
    print("=" * 100)
    print(
        f"FINAL RSM RESULTS — "
        f"{condition_name.upper()}"
    )
    print("=" * 100)

    print(
        f"N observations       : {n}"
    )

    print(
        f"Unique design points : "
        f"{unique_design_points}"
    )

    print(
        f"Model parameters     : {p}"
    )

    print(
        f"Residual df          : "
        f"{df_residual}"
    )

    print(
        f"Pure-error df        : "
        f"{df_pure_error}"
    )

    print(
        f"Lack-of-fit df       : "
        f"{df_lof}"
    )


    print("\nMODEL FIT")
    print("-" * 70)

    print(
        f"R²           = "
        f"{R2:.6f}"
    )

    print(
        f"Adjusted R²  = "
        f"{Adj_R2:.6f}"
    )

    print(
        f"Model F      = "
        f"{F_model:.6f}"
    )

    print(
        f"Model p      = "
        f"{p_model:.8g}"
    )


    print("\nCOEFFICIENTS")
    print("-" * 100)

    print(
        coefficient_table
        .round(
            {
                "Coefficient": 6,
                "SE": 6,
                "t": 4,
                "p": 6
            }
        )
        .to_string(
            index=False
        )
    )


    print("\nTERM ANOVA")
    print("-" * 100)

    print(
        term_table
        .round(
            {
                "SS": 4,
                "MS": 4,
                "Coefficient": 6,
                "F": 4,
                "p": 6
            }
        )
        .to_string(
            index=False
        )
    )


    print("\nOVERALL ANOVA / LACK OF FIT")
    print("-" * 100)

    print(
        anova
        .round(
            {
                "SS": 4,
                "MS": 4,
                "F": 4,
                "p": 8
            }
        )
        .to_string(
            index=False
        )
    )


    print("\nOPTIMUM WITHIN ORIGINAL BBD DOMAIN")
    print("-" * 70)

    print(
        optimum
        .round(4)
        .to_string(
            index=False
        )
    )


    # ========================================================
    # 13. EQUATION TO 2 DECIMAL PLACES
    # ========================================================

    signs = []

    for value in beta[1:]:

        if value >= 0:
            signs.append(
                f"+ {abs(value):.2f}"
            )

        else:
            signs.append(
                f"- {abs(value):.2f}"
            )


    equation = (
        f"Y = {beta[0]:.2f} "
        f"{signs[0]}A "
        f"{signs[1]}B "
        f"{signs[2]}C "
        f"{signs[3]}A² "
        f"{signs[4]}B² "
        f"{signs[5]}C² "
        f"{signs[6]}AB "
        f"{signs[7]}AC "
        f"{signs[8]}BC"
    )

    print("\nRSM EQUATION")
    print("-" * 70)

    print(
        equation
    )


    return {
        "data": d,
        "coefficients": coefficient_table,
        "terms": term_table,
        "anova": anova,
        "R2": R2,
        "Adjusted_R2": Adj_R2,
        "optimum": optimum,
        "equation": equation,
        "beta": beta,
        "predicted": y_pred
    }


# ============================================================
# 14. LOAD FINAL FILES
# ============================================================

raw_without = read_final_dataset(
    WITHOUT_BARK_FILE,
    suffix="X"
)

raw_with = read_final_dataset(
    WITH_BARK_FILE,
    suffix="O"
)


# ============================================================
# 15. EXTRACT BBD 1–15
# ============================================================

bbd_without = extract_bbd(
    raw_without,
    "without bark"
)

bbd_with = extract_bbd(
    raw_with,
    "with bark"
)


# ============================================================
# 16. TABLE 1 CHECK
# ============================================================

table1_without = make_table1_summary(
    bbd_without
)

table1_with = make_table1_summary(
    bbd_with
)


print("\n")
print("=" * 100)
print("TABLE 1 CHECK — WITHOUT BARK")
print("=" * 100)

print(
    table1_without[
        [
            "Run_number",
            "FPU",
            "Substrate",
            "Ratio",
            "Replicate_1",
            "Replicate_2",
            "Mean_SR",
            "SD_SR"
        ]
    ]
    .round(4)
    .to_string(index=False)
)


print("\n")
print("=" * 100)
print("TABLE 1 CHECK — WITH BARK")
print("=" * 100)

print(
    table1_with[
        [
            "Run_number",
            "FPU",
            "Substrate",
            "Ratio",
            "Replicate_1",
            "Replicate_2",
            "Mean_SR",
            "SD_SR"
        ]
    ]
    .round(4)
    .to_string(index=False)
)


# ============================================================
# 17. FINAL RSM FIT
# ============================================================

result_without = fit_rsm(
    bbd_without,
    "without bark"
)

result_with = fit_rsm(
    bbd_with,
    "with bark"
)


# ============================================================
# 18. FINAL MODEL SUMMARY
# ============================================================

model_summary = pd.DataFrame(
    {
        "Condition": [
            "Without bark",
            "With bark"
        ],

        "N": [
            len(bbd_without),
            len(bbd_with)
        ],

        "R2": [
            result_without["R2"],
            result_with["R2"]
        ],

        "Adjusted_R2": [
            result_without["Adjusted_R2"],
            result_with["Adjusted_R2"]
        ],

        "Optimum_FPU": [
            result_without[
                "optimum"
            ].iloc[0]["FPU"],

            result_with[
                "optimum"
            ].iloc[0]["FPU"]
        ],

        "Optimum_Substrate": [
            result_without[
                "optimum"
            ].iloc[0]["Substrate"],

            result_with[
                "optimum"
            ].iloc[0]["Substrate"]
        ],

        "Optimum_Ratio": [
            result_without[
                "optimum"
            ].iloc[0]["KMF006_ratio"],

            result_with[
                "optimum"
            ].iloc[0]["KMF006_ratio"]
        ],

        "Predicted_SR": [
            result_without[
                "optimum"
            ].iloc[0]["Predicted_SR"],

            result_with[
                "optimum"
            ].iloc[0]["Predicted_SR"]
        ]
    }
)


print("\n")
print("=" * 100)
print("FINAL RSM SUMMARY")
print("=" * 100)

print(
    model_summary
    .round(4)
    .to_string(index=False)
)


# ============================================================
# 19. OPTIONAL SAVE
# ============================================================

if SAVE_RESULTS:

    table1_without.to_csv(
        "RSM_Table1_without_bark.csv",
        index=False,
        encoding="utf-8-sig"
    )

    table1_with.to_csv(
        "RSM_Table1_with_bark.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result_without[
        "terms"
    ].to_csv(
        "RSM_terms_without_bark.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result_with[
        "terms"
    ].to_csv(
        "RSM_terms_with_bark.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result_without[
        "anova"
    ].to_csv(
        "RSM_ANOVA_without_bark.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result_with[
        "anova"
    ].to_csv(
        "RSM_ANOVA_with_bark.csv",
        index=False,
        encoding="utf-8-sig"
    )

    model_summary.to_csv(
        "RSM_final_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "\nResults saved."
    )


In [ ]:
# ============================================================
# FINAL FIGURE 2 — SIX SEPARATE RESPONSE-SURFACE PANELS
#
# FINAL RSM coefficients from n = 30 independent observations
#
# A = Enzyme loading (FPU/g-glucan)
# B = Substrate concentration (w/v, %)
# C = KMF006 blending ratio (v/v, %)
#
# Panels:
#   a: A × B, without bark, fixed C = 30%
#   b: A × B, with bark,    fixed C = 30%
#   c: A × C, without bark, fixed B = 11%
#   d: A × C, with bark,    fixed B = 11%
#   e: B × C, without bark, fixed A = 20 FPU/g-glucan
#   f: B × C, with bark,    fixed A = 20 FPU/g-glucan
#
# Each panel is generated separately for assembly in PowerPoint.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams


# ============================================================
# 1. GLOBAL STYLE
# ============================================================

rcParams["font.family"] = "Arial"
rcParams["font.size"] = 10

rcParams["axes.linewidth"] = 0.9

rcParams["xtick.direction"] = "out"
rcParams["ytick.direction"] = "out"

rcParams["xtick.major.width"] = 0.9
rcParams["ytick.major.width"] = 0.9

rcParams["xtick.major.size"] = 3.5
rcParams["ytick.major.size"] = 3.5


# ============================================================
# 2. SAVE OPTION
# ============================================================

SAVE_FIG = False

DPI = 600


# ============================================================
# 3. FINAL RSM EQUATIONS
#    Full-precision coefficients
# ============================================================

def predict_without_bark(A, B, C):

    return (
        61.26166667
        + 9.10006250 * A
        + 3.46212500 * B
        - 0.16418750 * C
        - 5.39720833 * A**2
        + 3.37541667 * B**2
        - 3.37770833 * C**2
        + 0.90412500 * A * B
        - 4.58825000 * A * C
        - 2.82787500 * B * C
    )


def predict_with_bark(A, B, C):

    return (
        53.73716667
        + 5.26806250 * A
        + 3.24918750 * B
        + 0.32325000 * C
        + 1.92385417 * A**2
        + 1.41160417 * B**2
        - 2.91977083 * C**2
        + 4.26637500 * A * B
        - 1.96750000 * A * C
        + 0.61475000 * B * C
    )


# ============================================================
# 4. CODED-VARIABLE FUNCTIONS
# ============================================================

def code_A(fpu):
    return (fpu - 20.0) / 10.0


def code_B(substrate):
    return (substrate - 11.0) / 2.0


def code_C(ratio):
    return (ratio - 30.0) / 20.0


# ============================================================
# 5. ORIGINAL BBD FACTOR RANGES
# ============================================================

FPU_VALUES = np.linspace(
    10,
    30,
    100
)

SUBSTRATE_VALUES = np.linspace(
    9,
    13,
    100
)

RATIO_VALUES = np.linspace(
    10,
    50,
    100
)


# ============================================================
# 6. SURFACE COLORS
# ============================================================

COLOR_AB = "#8DB5D2"   # blue
COLOR_AC = "#79B978"   # green
COLOR_BC = "#D99AA7"   # pink


# ============================================================
# 7. COMMON 3D STYLE
# ============================================================

def style_3d_axis(ax):

    # --------------------------------------------------------
    # Viewing angle
    # --------------------------------------------------------

    ax.view_init(
        elev=25,
        azim=-58
    )


    # --------------------------------------------------------
    # Pane appearance
    # --------------------------------------------------------

    ax.xaxis.pane.set_facecolor(
        (1, 1, 1, 0)
    )

    ax.yaxis.pane.set_facecolor(
        (1, 1, 1, 0)
    )

    ax.zaxis.pane.set_facecolor(
        (1, 1, 1, 0)
    )


    ax.xaxis.pane.set_edgecolor(
        "#777777"
    )

    ax.yaxis.pane.set_edgecolor(
        "#777777"
    )

    ax.zaxis.pane.set_edgecolor(
        "#777777"
    )


    # --------------------------------------------------------
    # Grid lines
    # --------------------------------------------------------

    for axis in [
        ax.xaxis,
        ax.yaxis,
        ax.zaxis
    ]:

        axis._axinfo[
            "grid"
        ][
            "linewidth"
        ] = 0.45

        axis._axinfo[
            "grid"
        ][
            "color"
        ] = (
            0.75,
            0.75,
            0.75,
            1
        )


    # --------------------------------------------------------
    # Tick labels
    # --------------------------------------------------------

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=9,
        pad=1
    )

    ax.tick_params(
        axis="z",
        which="major",
        labelsize=9,
        pad=1
    )


    # --------------------------------------------------------
    # Common z-axis
    # --------------------------------------------------------

    ax.set_zlim(
        35,
        80
    )

    ax.set_zticks(
        [
            40,
            50,
            60,
            70,
            80
        ]
    )


    ax.set_zlabel(
        "SR (%)",
        fontsize=11,
        fontweight="bold",
        labelpad=4
    )


# ============================================================
# 8. GENERAL PANEL FUNCTION
# ============================================================

def make_panel(
    X,
    Y,
    Z,
    color,
    xlabel,
    ylabel,
    xlim,
    ylim,
    xticks,
    yticks,
    filename
):

    # --------------------------------------------------------
    # Separate square-ish panel
    # --------------------------------------------------------

    fig = plt.figure(
        figsize=(4.8, 4.6)
    )


    # --------------------------------------------------------
    # [left, bottom, width, height]
    #
    # Extra bottom space prevents x/y labels from clipping.
    # Upper margin kept relatively small.
    # --------------------------------------------------------

    ax = fig.add_axes(
        [
            0.02,
            0.15,
            0.91,
            0.82
        ],
        projection="3d"
    )


    # --------------------------------------------------------
    # Surface
    # --------------------------------------------------------

    ax.plot_surface(
        X,
        Y,
        Z,

        color=color,

        edgecolor="#555555",

        linewidth=0.25,

        antialiased=True,

        alpha=0.95,

        rstride=3,
        cstride=3
    )


    # --------------------------------------------------------
    # Common style
    # --------------------------------------------------------

    style_3d_axis(
        ax
    )


    # --------------------------------------------------------
    # Limits
    # --------------------------------------------------------

    ax.set_xlim(
        *xlim
    )

    ax.set_ylim(
        *ylim
    )


    # --------------------------------------------------------
    # Ticks
    # --------------------------------------------------------

    ax.set_xticks(
        xticks
    )

    ax.set_yticks(
        yticks
    )


    # --------------------------------------------------------
    # Axis labels
    # --------------------------------------------------------

    ax.set_xlabel(
        xlabel,
        fontsize=11,
        fontweight="bold",
        labelpad=4
    )

    ax.set_ylabel(
        ylabel,
        fontsize=11,
        fontweight="bold",
        labelpad=5
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    if SAVE_FIG:

        plt.savefig(
            filename,
            dpi=DPI,
            bbox_inches="tight",
            pad_inches=0.18
        )


    plt.show()


# ============================================================
# 9. PANEL A
# WITHOUT BARK
# A × B
# Fixed C = 30%
# ============================================================

FPU_GRID, SUB_GRID = np.meshgrid(
    FPU_VALUES,
    SUBSTRATE_VALUES
)


A_GRID = code_A(
    FPU_GRID
)

B_GRID = code_B(
    SUB_GRID
)

C_FIXED = code_C(
    30
)


Z_A = predict_without_bark(
    A_GRID,
    B_GRID,
    C_FIXED
)


make_panel(

    X=FPU_GRID,

    Y=SUB_GRID,

    Z=Z_A,

    color=COLOR_AB,

    xlabel="Enzyme loading (A)",

    ylabel="Substrate conc. (B)",

    xlim=(
        10,
        30
    ),

    ylim=(
        9,
        13
    ),

    xticks=[
        10,
        15,
        20,
        25,
        30
    ],

    yticks=[
        9,
        10,
        11,
        12,
        13
    ],

    filename="Fig2_a_without_bark_AB.png"
)


# ============================================================
# 10. PANEL B
# WITH BARK
# A × B
# Fixed C = 30%
# ============================================================

Z_B = predict_with_bark(
    A_GRID,
    B_GRID,
    C_FIXED
)


make_panel(

    X=FPU_GRID,

    Y=SUB_GRID,

    Z=Z_B,

    color=COLOR_AB,

    xlabel="Enzyme loading (A)",

    ylabel="Substrate conc. (B)",

    xlim=(
        10,
        30
    ),

    ylim=(
        9,
        13
    ),

    xticks=[
        10,
        15,
        20,
        25,
        30
    ],

    yticks=[
        9,
        10,
        11,
        12,
        13
    ],

    filename="Fig2_b_with_bark_AB.png"
)


# ============================================================
# 11. PANEL C
# WITHOUT BARK
# A × C
# Fixed B = 11%
# ============================================================

FPU_GRID2, RATIO_GRID = np.meshgrid(
    FPU_VALUES,
    RATIO_VALUES
)


A_GRID2 = code_A(
    FPU_GRID2
)

B_FIXED = code_B(
    11
)

C_GRID = code_C(
    RATIO_GRID
)


Z_C = predict_without_bark(
    A_GRID2,
    B_FIXED,
    C_GRID
)


make_panel(

    X=FPU_GRID2,

    Y=RATIO_GRID,

    Z=Z_C,

    color=COLOR_AC,

    xlabel="Enzyme loading (A)",

    ylabel="KMF006 ratio (C)",

    xlim=(
        10,
        30
    ),

    ylim=(
        10,
        50
    ),

    xticks=[
        10,
        15,
        20,
        25,
        30
    ],

    yticks=[
        10,
        20,
        30,
        40,
        50
    ],

    filename="Fig2_c_without_bark_AC.png"
)


# ============================================================
# 12. PANEL D
# WITH BARK
# A × C
# Fixed B = 11%
# ============================================================

Z_D = predict_with_bark(
    A_GRID2,
    B_FIXED,
    C_GRID
)


make_panel(

    X=FPU_GRID2,

    Y=RATIO_GRID,

    Z=Z_D,

    color=COLOR_AC,

    xlabel="Enzyme loading (A)",

    ylabel="KMF006 ratio (C)",

    xlim=(
        10,
        30
    ),

    ylim=(
        10,
        50
    ),

    xticks=[
        10,
        15,
        20,
        25,
        30
    ],

    yticks=[
        10,
        20,
        30,
        40,
        50
    ],

    filename="Fig2_d_with_bark_AC.png"
)


# ============================================================
# 13. PANEL E
# WITHOUT BARK
# B × C
# Fixed A = 20 FPU/g-glucan
# ============================================================

SUB_GRID3, RATIO_GRID3 = np.meshgrid(
    SUBSTRATE_VALUES,
    RATIO_VALUES
)


A_FIXED = code_A(
    20
)

B_GRID3 = code_B(
    SUB_GRID3
)

C_GRID3 = code_C(
    RATIO_GRID3
)


Z_E = predict_without_bark(
    A_FIXED,
    B_GRID3,
    C_GRID3
)


make_panel(

    X=SUB_GRID3,

    Y=RATIO_GRID3,

    Z=Z_E,

    color=COLOR_BC,

    xlabel="Substrate conc. (B)",

    ylabel="KMF006 ratio (C)",

    xlim=(
        9,
        13
    ),

    ylim=(
        10,
        50
    ),

    xticks=[
        9,
        10,
        11,
        12,
        13
    ],

    yticks=[
        10,
        20,
        30,
        40,
        50
    ],

    filename="Fig2_e_without_bark_BC.png"
)


# ============================================================
# 14. PANEL F
# WITH BARK
# B × C
# Fixed A = 20 FPU/g-glucan
# ============================================================

Z_F = predict_with_bark(
    A_FIXED,
    B_GRID3,
    C_GRID3
)


make_panel(

    X=SUB_GRID3,

    Y=RATIO_GRID3,

    Z=Z_F,

    color=COLOR_BC,

    xlabel="Substrate conc. (B)",

    ylabel="KMF006 ratio (C)",

    xlim=(
        9,
        13
    ),

    ylim=(
        10,
        50
    ),

    xticks=[
        9,
        10,
        11,
        12,
        13
    ],

    yticks=[
        10,
        20,
        30,
        40,
        50
    ],

    filename="Fig2_f_with_bark_BC.png"
)


In [ ]:
# ============================================================
# FINAL ML DATASET AUDIT
#
# IMPORTANT EXPERIMENTAL STRUCTURE
# ------------------------------------------------------------
# Each row in the final Excel file represents:
#
#   1 independent saccharification experiment
#       +
#   mean of duplicate HPLC technical measurements
#
# Therefore:
#   - DO NOT average the two saccharification replicates.
#   - Each row remains one independent ML observation.
#   - Identical FPU/Substrate/Ratio conditions are assigned
#     to the same validation group to prevent train-test leakage.
# ============================================================

import re
import numpy as np
import pandas as pd


# ============================================================
# 1. FILE PATHS
# ============================================================

WITHOUT_BARK_FILE = "data/raw_without_bark.xlsx"
WITH_BARK_FILE    = "data/raw_with_bark.xlsx"


# ============================================================
# 2. OPTIONAL SAVE
# ============================================================

SAVE_CLEAN_DATA = False

WITHOUT_OUTPUT = "ML_final_without_bark.csv"
WITH_OUTPUT    = "ML_final_with_bark.csv"


# ============================================================
# 3. COMMON VARIABLE NAMES
# ============================================================

X_COLS = [
    "FPU",
    "Substrate",
    "Ratio"
]

TARGET = "Saccharification"


# ============================================================
# 4. READ FINAL MASTER DATA
# ============================================================

def read_final_ml_dataset(
    file_path,
    condition_name,
    expected_suffix
):

    # --------------------------------------------------------
    # Read Excel
    # --------------------------------------------------------

    df = pd.read_excel(
        file_path
    )


    print("\n")
    print("=" * 110)
    print(
        f"RAW FILE — {condition_name.upper()}"
    )
    print("=" * 110)

    print(
        "Original columns:",
        df.columns.tolist()
    )


    # --------------------------------------------------------
    # Rename columns
    # --------------------------------------------------------

    first_col = df.columns[0]

    # automatically detect saccharification column
    sac_cols = [
        col
        for col in df.columns
        if str(col).lower().startswith(
            "saccharificationrate"
        )
    ]


    if len(sac_cols) != 1:

        raise ValueError(
            "Could not uniquely identify "
            "the saccharification column.\n"
            f"Candidates: {sac_cols}"
        )


    sac_col = sac_cols[0]


    df = df.rename(
        columns={
            first_col:
                "Experimental_Run",

            "Enzymeratio":
                "Ratio",

            sac_col:
                "Saccharification"
        }
    )


    # --------------------------------------------------------
    # Required columns
    # --------------------------------------------------------

    required = [
        "Experimental_Run",
        "FPU",
        "Substrate",
        "Ratio",
        "Saccharification"
    ]


    missing = [
        col
        for col in required
        if col not in df.columns
    ]


    if missing:

        raise ValueError(
            f"Missing columns: {missing}"
        )


    df = df[
        required
    ].copy()


    # ========================================================
    # 5. CLEAN RUN IDs
    # ========================================================

    df[
        "Experimental_Run"
    ] = (
        df[
            "Experimental_Run"
        ]
        .astype(str)
        .str.strip()
    )


    # --------------------------------------------------------
    # Numeric run number
    # --------------------------------------------------------

    df[
        "Run_number"
    ] = pd.to_numeric(

        df[
            "Experimental_Run"
        ]
        .str.extract(
            r"(\d+)",
            expand=False
        ),

        errors="coerce"
    )


    # --------------------------------------------------------
    # Check X / O suffix
    # --------------------------------------------------------

    suffix_check = (

        df[
            "Experimental_Run"
        ]
        .str.endswith(
            expected_suffix
        )
    )


    if not suffix_check.all():

        print(
            "\nWARNING: unexpected run IDs:"
        )

        print(
            df.loc[
                ~suffix_check,
                "Experimental_Run"
            ]
            .unique()
        )


    # ========================================================
    # 6. NUMERIC CONVERSION
    # ========================================================

    for col in [
        "FPU",
        "Substrate",
        "Ratio",
        "Saccharification"
    ]:

        df[
            col
        ] = pd.to_numeric(
            df[
                col
            ],
            errors="coerce"
        )


    # --------------------------------------------------------
    # Missing-value check
    # --------------------------------------------------------

    if df[
        required
    ].isna().any().any():

        print(
            "\nRows containing missing values:"
        )

        print(
            df[
                df[
                    required
                ]
                .isna()
                .any(axis=1)
            ]
        )

        raise ValueError(
            "Missing values detected."
        )


    # ========================================================
    # 7. SORT DATA
    # ========================================================

    df = (

        df
        .sort_values(
            [
                "Run_number"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # 8. ASSIGN INDEPENDENT REPLICATE NUMBER
    # ========================================================
    #
    # Each Experimental_Run should contain:
    #
    # replicate 1
    # replicate 2
    #
    # These are independent saccharification experiments.
    # ========================================================

    df[
        "Experimental_Replicate"
    ] = (

        df
        .groupby(
            "Experimental_Run"
        )
        .cumcount()
        + 1
    )


    # --------------------------------------------------------
    # Observation ID
    # --------------------------------------------------------

    df[
        "Observation_ID"
    ] = (

        df[
            "Experimental_Run"
        ]

        + "_Rep"

        + df[
            "Experimental_Replicate"
        ].astype(str)
    )


    # ========================================================
    # 9. IDENTIFY BBD vs SUPPLEMENTARY DATA
    # ========================================================
    #
    # Runs 1–15 = original BBD
    # Runs >15  = supplementary experiments
    # ========================================================

    df[
        "Dataset_source"
    ] = np.where(

        df[
            "Run_number"
        ] <= 15,

        "BBD",

        "Supplementary"
    )


    # ========================================================
    # 10. CREATE LEAKAGE-CONTROL CONDITION GROUP
    # ========================================================
    #
    # Important:
    #
    # All observations with identical
    #
    #   FPU
    #   Substrate
    #   Ratio
    #
    # receive the SAME Condition_Group_ID.
    #
    # Therefore:
    #
    # 20 / 11 / 30
    #
    # from run 1, 5 and 12 all belong to the same group.
    #
    # These observations must NEVER be split between
    # train and test sets.
    # ========================================================

    df[
        "Condition_Group_ID"
    ] = (

        df
        .groupby(
            [
                "FPU",
                "Substrate",
                "Ratio"
            ],
            sort=True
        )
        .ngroup()
        + 1
    )


    # Add condition-specific prefix
    prefix = (
        "WB"
        if condition_name == "with bark"
        else "NB"
    )


    df[
        "Validation_Group"
    ] = (

        prefix
        + "_G"

        + df[
            "Condition_Group_ID"
        ]
        .astype(str)
    )


    df[
        "Bark_condition"
    ] = condition_name


    # ========================================================
    # 11. STRUCTURAL CHECK:
    #     TWO INDEPENDENT OBSERVATIONS PER RUN
    # ========================================================

    run_counts = (

        df
        .groupby(
            "Experimental_Run"
        )
        .size()
    )


    print("\n")
    print(
        "OBSERVATIONS PER EXPERIMENTAL RUN"
    )
    print("-" * 60)

    print(
        run_counts
        .value_counts()
        .sort_index()
        .to_string()
    )


    if not (
        run_counts == 2
    ).all():

        bad_runs = run_counts[
            run_counts != 2
        ]

        print(
            "\nWARNING:"
        )

        print(
            "The following runs do not contain "
            "exactly two independent observations:"
        )

        print(
            bad_runs
        )


    # ========================================================
    # 12. CHECK FACTOR CONSISTENCY WITHIN EACH RUN
    # ========================================================

    factor_check = (

        df
        .groupby(
            "Experimental_Run"
        )[
            X_COLS
        ]
        .nunique()
    )


    bad_factor_runs = factor_check[
        (
            factor_check > 1
        ).any(axis=1)
    ]


    if len(
        bad_factor_runs
    ) > 0:

        print(
            "\nERROR:"
        )

        print(
            "Different factor settings were found "
            "inside the same Experimental_Run."
        )

        print(
            bad_factor_runs
        )

        raise ValueError(
            "Factor inconsistency detected."
        )


    # ========================================================
    # 13. RUN-LEVEL SUMMARY
    # ========================================================

    run_summary = (

        df
        .groupby(
            [
                "Experimental_Run",
                "Run_number",
                "Dataset_source",
                "FPU",
                "Substrate",
                "Ratio",
                "Validation_Group"
            ],
            as_index=False
        )

        .agg(

            Replicate_1=(
                "Saccharification",
                "first"
            ),

            Replicate_2=(
                "Saccharification",
                "last"
            ),

            Mean_SR=(
                "Saccharification",
                "mean"
            ),

            SD_SR=(
                "Saccharification",
                "std"
            ),

            n_independent=(
                "Saccharification",
                "size"
            )
        )

        .sort_values(
            "Run_number"
        )

        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # 14. IDENTIFY REPEATED FACTOR CONDITIONS
    # ========================================================
    #
    # Example:
    #
    # 1X, 5X, 12X
    # all have 20 / 11 / 30
    #
    # These are separate experimental runs,
    # but must belong to the same CV group.
    # ========================================================

    condition_run_count = (

        run_summary
        .groupby(
            [
                "FPU",
                "Substrate",
                "Ratio"
            ]
        )[
            "Experimental_Run"
        ]
        .nunique()
        .reset_index(
            name="N_runs_same_condition"
        )
    )


    repeated_conditions = (

        condition_run_count[
            condition_run_count[
                "N_runs_same_condition"
            ] > 1
        ]
    )


    repeated_detail = (

        run_summary
        .merge(
            repeated_conditions[
                [
                    "FPU",
                    "Substrate",
                    "Ratio"
                ]
            ],

            on=[
                "FPU",
                "Substrate",
                "Ratio"
            ],

            how="inner"
        )

        .sort_values(
            [
                "FPU",
                "Substrate",
                "Ratio",
                "Run_number"
            ]
        )
    )


    # ========================================================
    # 15. DATASET SUMMARY
    # ========================================================

    n_obs = len(
        df
    )

    n_runs = df[
        "Experimental_Run"
    ].nunique()

    n_groups = df[
        "Validation_Group"
    ].nunique()


    n_bbd_obs = (
        df[
            "Dataset_source"
        ] == "BBD"
    ).sum()


    n_supp_obs = (
        df[
            "Dataset_source"
        ] == "Supplementary"
    ).sum()


    n_bbd_runs = (
        run_summary[
            "Dataset_source"
        ] == "BBD"
    ).sum()


    n_supp_runs = (
        run_summary[
            "Dataset_source"
        ] == "Supplementary"
    ).sum()


    # ========================================================
    # 16. DOMAIN SUMMARY
    # ========================================================

    domain = pd.DataFrame(
        {
            "Variable": [
                "Enzyme loading (FPU)",
                "Substrate concentration (%)",
                "KMF006 ratio (%)",
                "Saccharification (%)"
            ],

            "Minimum": [
                df[
                    "FPU"
                ].min(),

                df[
                    "Substrate"
                ].min(),

                df[
                    "Ratio"
                ].min(),

                df[
                    "Saccharification"
                ].min()
            ],

            "Maximum": [
                df[
                    "FPU"
                ].max(),

                df[
                    "Substrate"
                ].max(),

                df[
                    "Ratio"
                ].max(),

                df[
                    "Saccharification"
                ].max()
            ]
        }
    )


    # ========================================================
    # 17. BBD-ONLY DOMAIN
    # ========================================================

    bbd_df = df[
        df[
            "Dataset_source"
        ] == "BBD"
    ]


    bbd_domain = pd.DataFrame(
        {
            "Variable": [
                "FPU",
                "Substrate",
                "Ratio"
            ],

            "Minimum": [
                bbd_df[
                    "FPU"
                ].min(),

                bbd_df[
                    "Substrate"
                ].min(),

                bbd_df[
                    "Ratio"
                ].min()
            ],

            "Maximum": [
                bbd_df[
                    "FPU"
                ].max(),

                bbd_df[
                    "Substrate"
                ].max(),

                bbd_df[
                    "Ratio"
                ].max()
            ]
        }
    )


    # ========================================================
    # 18. SUPPLEMENTARY DOMAIN
    # ========================================================

    supp_df = df[
        df[
            "Dataset_source"
        ] == "Supplementary"
    ]


    supp_domain = pd.DataFrame(
        {
            "Variable": [
                "FPU",
                "Substrate",
                "Ratio"
            ],

            "Minimum": [
                supp_df[
                    "FPU"
                ].min(),

                supp_df[
                    "Substrate"
                ].min(),

                supp_df[
                    "Ratio"
                ].min()
            ],

            "Maximum": [
                supp_df[
                    "FPU"
                ].max(),

                supp_df[
                    "Substrate"
                ].max(),

                supp_df[
                    "Ratio"
                ].max()
            ]
        }
    )


    # ========================================================
    # 19. PRINT AUDIT RESULTS
    # ========================================================

    print("\n")
    print("=" * 110)
    print(
        f"FINAL ML DATASET AUDIT — "
        f"{condition_name.upper()}"
    )
    print("=" * 110)


    print(
        f"Total ML observations       : {n_obs}"
    )

    print(
        f"Experimental runs           : {n_runs}"
    )

    print(
        f"Unique condition groups     : {n_groups}"
    )

    print(
        f"BBD observations            : {n_bbd_obs}"
    )

    print(
        f"BBD experimental runs       : {n_bbd_runs}"
    )

    print(
        f"Supplementary observations  : {n_supp_obs}"
    )

    print(
        f"Supplementary runs          : {n_supp_runs}"
    )


    # --------------------------------------------------------
    # Full domain
    # --------------------------------------------------------

    print("\n")
    print(
        "FULL ML EXPERIMENTAL DOMAIN"
    )
    print("-" * 65)

    print(
        domain
        .round(4)
        .to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # BBD domain
    # --------------------------------------------------------

    print("\n")
    print(
        "ORIGINAL BBD DOMAIN"
    )
    print("-" * 65)

    print(
        bbd_domain
        .round(4)
        .to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Supplementary domain
    # --------------------------------------------------------

    print("\n")
    print(
        "SUPPLEMENTARY EXPERIMENTAL DOMAIN"
    )
    print("-" * 65)

    print(
        supp_domain
        .round(4)
        .to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Repeated conditions
    # --------------------------------------------------------

    print("\n")
    print(
        "REPEATED FACTOR CONDITIONS"
    )
    print(
        "(All observations within each condition "
        "must stay in the same CV fold)"
    )
    print("-" * 110)


    if len(
        repeated_detail
    ) == 0:

        print(
            "None"
        )

    else:

        print(
            repeated_detail[
                [
                    "Experimental_Run",
                    "Dataset_source",
                    "FPU",
                    "Substrate",
                    "Ratio",
                    "Replicate_1",
                    "Replicate_2",
                    "Mean_SR",
                    "Validation_Group"
                ]
            ]
            .round(4)
            .to_string(
                index=False
            )
        )


    # --------------------------------------------------------
    # Full run summary
    # --------------------------------------------------------

    print("\n")
    print(
        "RUN-LEVEL SUMMARY"
    )
    print("-" * 110)

    print(
        run_summary[
            [
                "Experimental_Run",
                "Dataset_source",
                "FPU",
                "Substrate",
                "Ratio",
                "Replicate_1",
                "Replicate_2",
                "Mean_SR",
                "SD_SR",
                "Validation_Group"
            ]
        ]
        .round(4)
        .to_string(
            index=False
        )
    )


    # ========================================================
    # 20. FINAL ML TABLE
    # ========================================================
    #
    # No averaging is performed here.
    #
    # Every row remains an independent experimental
    # observation.
    # ========================================================

    ml_data = df[
        [
            "Observation_ID",
            "Experimental_Run",
            "Experimental_Replicate",
            "Bark_condition",
            "Dataset_source",
            "Validation_Group",
            "FPU",
            "Substrate",
            "Ratio",
            "Saccharification"
        ]
    ].copy()


    return {
        "ml_data":
            ml_data,

        "run_summary":
            run_summary,

        "domain":
            domain,

        "bbd_domain":
            bbd_domain,

        "supplementary_domain":
            supp_domain,

        "repeated_conditions":
            repeated_detail
    }


# ============================================================
# 21. WITHOUT BARK
# ============================================================

result_without = read_final_ml_dataset(

    file_path=WITHOUT_BARK_FILE,

    condition_name="without bark",

    expected_suffix="X"
)


ml_without = result_without[
    "ml_data"
]


# ============================================================
# 22. WITH BARK
# ============================================================

result_with = read_final_ml_dataset(

    file_path=WITH_BARK_FILE,

    condition_name="with bark",

    expected_suffix="O"
)


ml_with = result_with[
    "ml_data"
]


# ============================================================
# 23. FINAL COMBINED AUDIT SUMMARY
# ============================================================

summary = pd.DataFrame(
    {
        "Condition": [
            "Without bark",
            "With bark"
        ],

        "ML_observations": [
            len(
                ml_without
            ),

            len(
                ml_with
            )
        ],

        "Experimental_runs": [
            ml_without[
                "Experimental_Run"
            ].nunique(),

            ml_with[
                "Experimental_Run"
            ].nunique()
        ],

        "Unique_condition_groups": [
            ml_without[
                "Validation_Group"
            ].nunique(),

            ml_with[
                "Validation_Group"
            ].nunique()
        ],

        "BBD_observations": [
            (
                ml_without[
                    "Dataset_source"
                ] == "BBD"
            ).sum(),

            (
                ml_with[
                    "Dataset_source"
                ] == "BBD"
            ).sum()
        ],

        "Supplementary_observations": [
            (
                ml_without[
                    "Dataset_source"
                ] == "Supplementary"
            ).sum(),

            (
                ml_with[
                    "Dataset_source"
                ] == "Supplementary"
            ).sum()
        ]
    }
)


print("\n")
print("=" * 110)
print(
    "FINAL COMBINED ML AUDIT"
)
print("=" * 110)

print(
    summary
    .to_string(
        index=False
    )
)


# ============================================================
# 24. HARD ASSERTIONS
# ============================================================
#
# These values are expected from the current master files.
# If any assertion fails, STOP before ML modeling.
# ============================================================

assert len(
    ml_without
) == 54, (
    "Without-bark observation count is not 54."
)


assert len(
    ml_with
) == 68, (
    "With-bark observation count is not 68."
)


assert ml_without[
    "Experimental_Run"
].nunique() == 27, (
    "Without-bark experimental-run count is not 27."
)


assert ml_with[
    "Experimental_Run"
].nunique() == 34, (
    "With-bark experimental-run count is not 34."
)


assert ml_without[
    "Validation_Group"
].nunique() == 23, (
    "Without-bark unique condition-group count is not 23."
)


assert ml_with[
    "Validation_Group"
].nunique() == 29, (
    "With-bark unique condition-group count is not 29."
)


assert (
    ml_without[
        "Dataset_source"
    ] == "BBD"
).sum() == 30


assert (
    ml_with[
        "Dataset_source"
    ] == "BBD"
).sum() == 30


print("\n")
print("=" * 110)
print(
    "AUDIT STATUS: PASS"
)
print("=" * 110)

print(
    "The final ML datasets are structurally ready "
    "for leakage-controlled model validation."
)


# ============================================================
# 25. OPTIONAL SAVE
# ============================================================

if SAVE_CLEAN_DATA:

    ml_without.to_csv(
        WITHOUT_OUTPUT,
        index=False,
        encoding="utf-8-sig"
    )

    ml_with.to_csv(
        WITH_OUTPUT,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nSaved:")
    print(
        WITHOUT_OUTPUT
    )
    print(
        WITH_OUTPUT
    )


In [ ]:
# ============================================================
# CONDITION-LEVEL DATASET FOR REVIEWER 3
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Reviewer 3 requested that all independent experiments
# performed at the same factor combination be aggregated
# before cross-validation.
#
# Grouping unit:
#   FPU + Substrate concentration + KMF006 blending ratio
#
# Expected result:
#   Without bark: 54 observations -> 23 unique conditions
#   With bark   : 68 observations -> 29 unique conditions
#
# IMPORTANT
# ------------------------------------------------------------
# This cell ONLY creates and audits the condition-level dataset.
# No machine-learning model is fitted here.
# ============================================================


import numpy as np
import pandas as pd


# ============================================================
# 1. SETTINGS
# ============================================================

FEATURES = [
    "FPU",
    "Substrate",
    "Ratio"
]

TARGET = "Saccharification"


# ============================================================
# 2. CHECK REQUIRED DATA
# ============================================================

for object_name in [
    "ml_without",
    "ml_with"
]:

    if object_name not in globals():

        raise RuntimeError(
            f"{object_name} not found.\n"
            "Run the FINAL ML DATASET AUDIT cell first."
        )


required_columns = [
    "Experimental_Run",
    "Dataset_source",
    "FPU",
    "Substrate",
    "Ratio",
    "Saccharification"
]


for data_name, data in [
    ("ml_without", ml_without),
    ("ml_with", ml_with)
]:

    missing = [
        col
        for col in required_columns
        if col not in data.columns
    ]

    if missing:

        raise ValueError(
            f"{data_name} missing columns:\n"
            f"{missing}"
        )


# ============================================================
# 3. CREATE CONDITION-LEVEL DATASET
# ============================================================

def create_condition_level_dataset(
    data,
    condition_name
):

    data = data.copy()


    # --------------------------------------------------------
    # Create explicit unique condition ID
    # --------------------------------------------------------

    data["Condition_ID"] = (

        "FPU_"
        + data["FPU"].astype(str)

        + "_SUB_"
        + data["Substrate"].astype(str)

        + "_RATIO_"
        + data["Ratio"].astype(str)
    )


    # --------------------------------------------------------
    # Audit number of raw observations and runs
    # --------------------------------------------------------

    n_observations = len(data)

    n_runs = data[
        "Experimental_Run"
    ].nunique()

    n_conditions = data[
        "Condition_ID"
    ].nunique()


    # --------------------------------------------------------
    # Aggregate ALL independent observations sharing
    # the identical factor combination
    # --------------------------------------------------------

    condition_level = (

        data

        .groupby(
            FEATURES,
            as_index=False
        )

        .agg(

            Saccharification=(
                TARGET,
                "mean"
            ),

            Saccharification_SD=(
                TARGET,
                "std"
            ),

            N_observations=(
                TARGET,
                "size"
            ),

            N_experimental_runs=(
                "Experimental_Run",
                "nunique"
            ),

            Experimental_runs=(
                "Experimental_Run",
                lambda x:
                    ", ".join(
                        sorted(
                            x.astype(str).unique()
                        )
                    )
            ),

            Dataset_sources=(
                "Dataset_source",
                lambda x:
                    ", ".join(
                        sorted(
                            x.astype(str).unique()
                        )
                    )
            )
        )
    )


    # --------------------------------------------------------
    # Re-create condition ID after aggregation
    # --------------------------------------------------------

    condition_level["Condition_ID"] = (

        "FPU_"
        + condition_level["FPU"].astype(str)

        + "_SUB_"
        + condition_level["Substrate"].astype(str)

        + "_RATIO_"
        + condition_level["Ratio"].astype(str)
    )


    # --------------------------------------------------------
    # Put columns in readable order
    # --------------------------------------------------------

    condition_level = condition_level[
        [
            "Condition_ID",
            "FPU",
            "Substrate",
            "Ratio",
            "Saccharification",
            "Saccharification_SD",
            "N_observations",
            "N_experimental_runs",
            "Experimental_runs",
            "Dataset_sources"
        ]
    ].copy()


    # --------------------------------------------------------
    # SD is undefined when only one observation exists.
    # Keep NaN rather than forcing SD = 0.
    # --------------------------------------------------------


    # --------------------------------------------------------
    # Audit duplicate predictor combinations
    # --------------------------------------------------------

    duplicate_conditions = (

        condition_level

        .duplicated(
            subset=FEATURES,
            keep=False
        )

        .sum()
    )


    print("\n")
    print("=" * 100)
    print(
        f"CONDITION-LEVEL DATASET: {condition_name.upper()}"
    )
    print("=" * 100)

    print(
        f"Raw observations       : {n_observations}"
    )

    print(
        f"Experimental runs      : {n_runs}"
    )

    print(
        f"Unique factor conditions: {n_conditions}"
    )

    print(
        f"Final condition rows   : {len(condition_level)}"
    )

    print(
        f"Duplicate conditions   : {duplicate_conditions}"
    )


    print("\nCondition-level dataset:\n")

    print(
        condition_level
        .to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Hard assertion
    # --------------------------------------------------------

    assert duplicate_conditions == 0, (
        "Duplicate factor combinations remain "
        "after condition-level aggregation."
    )


    return condition_level


# ============================================================
# 4. WITHOUT BARK
# ============================================================

condition_without = create_condition_level_dataset(
    ml_without,
    "without bark"
)


# ============================================================
# 5. WITH BARK
# ============================================================

condition_with = create_condition_level_dataset(
    ml_with,
    "with bark"
)


# ============================================================
# 6. EXPECTED SAMPLE COUNTS
# ============================================================

assert len(
    condition_without
) == 23, (
    f"Expected 23 unique conditions without bark, "
    f"but obtained {len(condition_without)}."
)


assert len(
    condition_with
) == 29, (
    f"Expected 29 unique conditions with bark, "
    f"but obtained {len(condition_with)}."
)


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

summary = pd.DataFrame(
    {
        "Condition": [
            "Without bark",
            "With bark"
        ],

        "Original_observations": [
            len(ml_without),
            len(ml_with)
        ],

        "Experimental_runs": [
            ml_without[
                "Experimental_Run"
            ].nunique(),

            ml_with[
                "Experimental_Run"
            ].nunique()
        ],

        "Unique_factor_conditions": [
            len(condition_without),
            len(condition_with)
        ]
    }
)


print("\n")
print("=" * 100)
print("REVIEWER 3 CONDITION-LEVEL AGGREGATION SUMMARY")
print("=" * 100)

print(
    summary
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 100)
print("STATUS: PASS")
print("=" * 100)

print(
    "All observations sharing identical FPU, substrate "
    "concentration, and KMF006 blending ratio have been "
    "aggregated before model validation."
)


In [ ]:
# ============================================================
# REVIEWER 3 — FULLY NESTED CONDITION-LEVEL VALIDATION
#
# PURPOSE
# ------------------------------------------------------------
# Resolve Reviewer 3 concerns regarding:
#
# 1. independence of the validation unit
# 2. hyperparameter-selection leakage
# 3. preprocessing leakage
#
#
# DATA UNIT
# ------------------------------------------------------------
# BEFORE cross-validation, all independent observations
# sharing the same:
#
#   FPU + Substrate + KMF006 ratio
#
# were aggregated into one condition-level response.
#
# WITHOUT BARK : 23 unique conditions
# WITH BARK    : 29 unique conditions
#
#
# OUTER CV
# ------------------------------------------------------------
# Leave-One-Condition-Out CV
#
# Each unique process condition is held out exactly once.
#
#
# INNER CV
# ------------------------------------------------------------
# 5-fold K-fold CV performed ONLY on the outer-training data.
#
# Hyperparameters and all data-dependent preprocessing
# are selected exclusively within the outer-training set.
#
#
# OPTIMIZATION METRIC
# ------------------------------------------------------------
# RMSE
#
#
# MODELS
# ------------------------------------------------------------
# 1. Quadratic Ridge
# 2. SVR
# 3. Gaussian Process Regression (GPR)
# 4. Spline Ridge
#
#
# IMPORTANT
# ------------------------------------------------------------
# The OUTER held-out condition is NEVER used for:
#
#   - X scaling
#   - y scaling
#   - polynomial transformation fitting
#   - spline-knot estimation
#   - hyperparameter selection
#   - kernel-family selection
#   - model fitting
#
# Only the pooled OUTER predictions are used to calculate
# the final R2, RMSE, MAE, and MRE.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import warnings
import time
import numpy as np
import pandas as pd

from collections import Counter

from sklearn.base import clone

from sklearn.model_selection import (
    LeaveOneOut,
    KFold,
    GridSearchCV
)

from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    MinMaxScaler,
    PolynomialFeatures,
    SplineTransformer
)

from sklearn.pipeline import Pipeline

from sklearn.compose import (
    TransformedTargetRegressor
)

from sklearn.linear_model import Ridge
from sklearn.svm import SVR

from sklearn.gaussian_process import (
    GaussianProcessRegressor
)

from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    Matern,
    RationalQuadratic,
    WhiteKernel
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)


warnings.filterwarnings("ignore")


# ============================================================
# 1. SETTINGS
# ============================================================

FEATURES = [
    "FPU",
    "Substrate",
    "Ratio"
]

TARGET = "Saccharification"

RANDOM_STATE = 2026

INNER_FOLDS = 5

N_JOBS = -1


# ============================================================
# 2. CHECK CONDITION-LEVEL DATASETS
# ============================================================

for object_name in [
    "condition_without",
    "condition_with"
]:

    if object_name not in globals():

        raise RuntimeError(
            f"{object_name} not found.\n"
            "Run the condition-level aggregation cell first."
        )


assert len(condition_without) == 23
assert len(condition_with) == 29


for dataset_name, dataset in [

    (
        "WITHOUT BARK",
        condition_without
    ),

    (
        "WITH BARK",
        condition_with
    )

]:

    duplicate_n = (

        dataset
        .duplicated(
            subset=FEATURES,
            keep=False
        )
        .sum()
    )

    assert duplicate_n == 0, (
        f"{dataset_name}: "
        "duplicate factor conditions remain."
    )


print("=" * 100)
print("DATA CHECK")
print("=" * 100)

print(
    "Without bark:",
    len(condition_without),
    "unique conditions"
)

print(
    "With bark   :",
    len(condition_with),
    "unique conditions"
)

print(
    "Duplicate predictor conditions: 0 / 0"
)


# ============================================================
# 3. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )


    mse = mean_squared_error(
        y_true,
        y_pred
    )


    rmse = np.sqrt(
        mse
    )


    mae = mean_absolute_error(
        y_true,
        y_pred
    )


    r2 = r2_score(
        y_true,
        y_pred
    )


    mre = np.mean(

        np.abs(
            y_true
            -
            y_pred
        )

        /

        np.maximum(
            np.abs(
                y_true
            ),
            1e-12
        )
    )


    return {

        "R2":
            r2,

        "RMSE":
            rmse,

        "MAE":
            mae,

        "MRE":
            mre
    }


# ============================================================
# 4. GPR KERNEL FACTORY
# ============================================================

def make_gpr_kernels():

    kernels = []


    # --------------------------------------------------------
    # RBF
    # --------------------------------------------------------

    kernels.append(

        ConstantKernel(

            constant_value=1.0,

            constant_value_bounds=(
                0.05,
                20.0
            )
        )

        *

        RBF(

            length_scale=np.ones(
                3
            ),

            length_scale_bounds=(
                0.05,
                20.0
            )
        )

        +

        WhiteKernel(

            noise_level=0.05,

            noise_level_bounds=(
                1e-4,
                2.0
            )
        )
    )


    # --------------------------------------------------------
    # Matern nu = 1.5
    # --------------------------------------------------------

    kernels.append(

        ConstantKernel(

            constant_value=1.0,

            constant_value_bounds=(
                0.05,
                20.0
            )
        )

        *

        Matern(

            length_scale=np.ones(
                3
            ),

            length_scale_bounds=(
                0.05,
                20.0
            ),

            nu=1.5
        )

        +

        WhiteKernel(

            noise_level=0.05,

            noise_level_bounds=(
                1e-4,
                2.0
            )
        )
    )


    # --------------------------------------------------------
    # Matern nu = 2.5
    # --------------------------------------------------------

    kernels.append(

        ConstantKernel(

            constant_value=1.0,

            constant_value_bounds=(
                0.05,
                20.0
            )
        )

        *

        Matern(

            length_scale=np.ones(
                3
            ),

            length_scale_bounds=(
                0.05,
                20.0
            ),

            nu=2.5
        )

        +

        WhiteKernel(

            noise_level=0.05,

            noise_level_bounds=(
                1e-4,
                2.0
            )
        )
    )


    # --------------------------------------------------------
    # Rational Quadratic
    # --------------------------------------------------------

    kernels.append(

        ConstantKernel(

            constant_value=1.0,

            constant_value_bounds=(
                0.05,
                20.0
            )
        )

        *

        RationalQuadratic(

            length_scale=1.0,

            alpha=1.0,

            length_scale_bounds=(
                0.05,
                20.0
            ),

            alpha_bounds=(
                0.05,
                20.0
            )
        )

        +

        WhiteKernel(

            noise_level=0.05,

            noise_level_bounds=(
                1e-4,
                2.0
            )
        )
    )


    return kernels


# ============================================================
# 5. MODEL + SEARCH-SPACE FACTORY
# ============================================================

def make_model_and_grid(
    model_name
):

    # ========================================================
    # QUADRATIC RIDGE
    # ========================================================

    if model_name == "Quadratic":

        regression_pipeline = Pipeline(

            steps=[

                (
                    "x_scaler",
                    StandardScaler()
                ),

                (
                    "poly",
                    PolynomialFeatures(

                        degree=2,

                        include_bias=False
                    )
                ),

                (
                    "poly_scaler",
                    StandardScaler()
                ),

                (
                    "ridge",
                    Ridge()
                )
            ]
        )


        model = TransformedTargetRegressor(

            regressor=
                regression_pipeline,

            transformer=
                StandardScaler()
        )


        param_grid = {

            "regressor__x_scaler": [

                StandardScaler(),

                RobustScaler(
                    quantile_range=(
                        25,
                        75
                    )
                ),

                MinMaxScaler()
            ],

            "regressor__ridge__alpha": [

                0.0001,
                0.001,
                0.003,
                0.01,
                0.03,
                0.10,
                0.30,
                1.0,
                3.0,
                10.0
            ]
        }


    # ========================================================
    # SUPPORT VECTOR REGRESSION
    # ========================================================

    elif model_name == "SVR":

        regression_pipeline = Pipeline(

            steps=[

                (
                    "x_scaler",
                    StandardScaler()
                ),

                (
                    "svr",
                    SVR(
                        kernel="rbf"
                    )
                )
            ]
        )


        model = TransformedTargetRegressor(

            regressor=
                regression_pipeline,

            transformer=
                StandardScaler()
        )


        param_grid = {

            "regressor__x_scaler": [

                StandardScaler(),

                RobustScaler(
                    quantile_range=(
                        25,
                        75
                    )
                ),

                MinMaxScaler()
            ],

            "regressor__svr__C": [

                0.3,
                1,
                3,
                10,
                30,
                100,
                300,
                1000
            ],

            "regressor__svr__gamma": [

                0.003,
                0.01,
                0.03,
                0.10,
                0.30
            ],

            "regressor__svr__epsilon": [

                0.01,
                0.03,
                0.05,
                0.10,
                0.20
            ]
        }


    # ========================================================
    # SPLINE RIDGE
    # ========================================================

    elif model_name == "SplineRidge":

        regression_pipeline = Pipeline(

            steps=[

                (
                    "x_scaler",
                    StandardScaler()
                ),

                (
                    "spline",
                    SplineTransformer(

                        knots="quantile",

                        extrapolation="linear",

                        include_bias=False
                    )
                ),

                (
                    "basis_scaler",
                    StandardScaler()
                ),

                (
                    "ridge",
                    Ridge()
                )
            ]
        )


        model = TransformedTargetRegressor(

            regressor=
                regression_pipeline,

            transformer=
                StandardScaler()
        )


        param_grid = {

            "regressor__x_scaler": [

                StandardScaler(),

                RobustScaler(
                    quantile_range=(
                        25,
                        75
                    )
                ),

                MinMaxScaler()
            ],

            "regressor__spline__n_knots": [

                3,
                4,
                5,
                6
            ],

            "regressor__spline__degree": [

                2,
                3
            ],

            "regressor__ridge__alpha": [

                0.001,
                0.01,
                0.03,
                0.10,
                0.30,
                1.0,
                3.0
            ]
        }


    # ========================================================
    # GAUSSIAN PROCESS REGRESSION
    # ========================================================

    elif model_name == "GPR":

        regression_pipeline = Pipeline(

            steps=[

                (
                    "x_scaler",
                    StandardScaler()
                ),

                (
                    "gpr",
                    GaussianProcessRegressor(

                        alpha=1e-6,

                        normalize_y=False,

                        optimizer="fmin_l_bfgs_b",

                        n_restarts_optimizer=1,

                        random_state=
                            RANDOM_STATE
                    )
                )
            ]
        )


        model = TransformedTargetRegressor(

            regressor=
                regression_pipeline,

            transformer=
                StandardScaler()
        )


        param_grid = {

            "regressor__x_scaler": [

                StandardScaler(),

                RobustScaler(
                    quantile_range=(
                        25,
                        75
                    )
                )
            ],

            "regressor__gpr__kernel":
                make_gpr_kernels()
        }


    else:

        raise ValueError(
            f"Unknown model: "
            f"{model_name}"
        )


    return (
        model,
        param_grid
    )


# ============================================================
# 6. PARAMETER DISPLAY HELPERS
# ============================================================

def scaler_name(
    scaler
):

    if isinstance(
        scaler,
        StandardScaler
    ):

        return "Standard"


    if isinstance(
        scaler,
        RobustScaler
    ):

        return "Robust"


    if isinstance(
        scaler,
        MinMaxScaler
    ):

        return "MinMax"


    return str(
        scaler
    )


def kernel_family_name(
    kernel
):

    text = str(
        kernel
    )


    if "RationalQuadratic" in text:

        return "RationalQuadratic"


    if "Matern" in text:

        if "nu=1.5" in text:

            return "Matern15"

        if "nu=2.5" in text:

            return "Matern25"

        return "Matern"


    if "RBF" in text:

        return "RBF"


    return text


def clean_best_params(
    model_name,
    params
):

    cleaned = {}


    for key, value in params.items():

        short_key = (
            key
            .replace(
                "regressor__",
                ""
            )
            .replace(
                "x_scaler",
                "scaler"
            )
            .replace(
                "svr__",
                ""
            )
            .replace(
                "spline__",
                ""
            )
            .replace(
                "ridge__",
                ""
            )
            .replace(
                "gpr__",
                ""
            )
        )


        if short_key == "scaler":

            value = scaler_name(
                value
            )


        elif short_key == "kernel":

            value = kernel_family_name(
                value
            )


        cleaned[
            short_key
        ] = value


    return cleaned


# ============================================================
# 7. NESTED CONDITION-LEVEL LOOCV
# ============================================================

MODEL_LIST = [

    "Quadratic",

    "SVR",

    "GPR",

    "SplineRidge"
]


def run_fully_nested_condition_loocv(
    data,
    condition_name
):

    print("\n")
    print("=" * 120)

    print(
        "FULLY NESTED CONDITION-LEVEL LOOCV — "
        f"{condition_name.upper()}"
    )

    print("=" * 120)


    # --------------------------------------------------------
    # Prepare X / y
    # --------------------------------------------------------

    X = (

        data[
            FEATURES
        ]

        .to_numpy(
            dtype=float
        )
    )


    y = (

        data[
            TARGET
        ]

        .to_numpy(
            dtype=float
        )
    )


    condition_ids = (

        data[
            "Condition_ID"
        ]

        .astype(str)

        .to_numpy()
    )


    n_samples = len(
        data
    )


    print(
        f"N unique conditions : {n_samples}"
    )

    print(
        f"Outer CV            : {n_samples}-fold "
        "Leave-One-Condition-Out"
    )

    print(
        f"Inner CV            : {INNER_FOLDS}-fold KFold"
    )

    print(
        "Inner optimization  : RMSE"
    )


    # --------------------------------------------------------
    # Storage
    # --------------------------------------------------------

    predictions = {

        model_name:
            np.full(
                n_samples,
                np.nan,
                dtype=float
            )

        for model_name in MODEL_LIST
    }


    parameter_history = {

        model_name:
            []

        for model_name in MODEL_LIST
    }


    prediction_rows = []


    outer_cv = LeaveOneOut()


    start_time = time.time()


    # ========================================================
    # OUTER LOOP
    # ========================================================

    for outer_fold, (
        train_idx,
        test_idx
    ) in enumerate(
        outer_cv.split(
            X
        ),
        start=1
    ):


        X_train = X[
            train_idx
        ]


        y_train = y[
            train_idx
        ]


        X_test = X[
            test_idx
        ]


        y_test = y[
            test_idx
        ]


        held_out_id = (
            condition_ids[
                test_idx[
                    0
                ]
            ]
        )


        # ----------------------------------------------------
        # Explicit leakage audit
        # ----------------------------------------------------

        held_out_tuple = tuple(
            X_test[
                0
            ]
        )


        training_tuples = {

            tuple(
                row
            )

            for row in X_train
        }


        if held_out_tuple in training_tuples:

            raise RuntimeError(
                "LEAKAGE DETECTED: "
                f"{held_out_id} is present "
                "in the outer-training data."
            )


        # ----------------------------------------------------
        # INNER CV
        #
        # Only outer-training samples are available here.
        # ----------------------------------------------------

        inner_cv = KFold(

            n_splits=
                INNER_FOLDS,

            shuffle=True,

            random_state=
                RANDOM_STATE
                +
                outer_fold
        )


        fold_predictions = {}


        # ====================================================
        # MODEL LOOP
        # ====================================================

        for model_name in MODEL_LIST:


            base_model, param_grid = (
                make_model_and_grid(
                    model_name
                )
            )


            search = GridSearchCV(

                estimator=
                    base_model,

                param_grid=
                    param_grid,

                scoring=
                    "neg_root_mean_squared_error",

                cv=
                    inner_cv,

                refit=True,

                n_jobs=
                    N_JOBS,

                error_score=
                    np.nan,

                return_train_score=False
            )


            # ------------------------------------------------
            # Everything here sees OUTER TRAIN only
            # ------------------------------------------------

            search.fit(

                X_train,

                y_train
            )


            # ------------------------------------------------
            # One completely unseen condition
            # ------------------------------------------------

            pred = float(

                np.asarray(

                    search.predict(
                        X_test
                    )

                ).ravel()[
                    0
                ]
            )


            predictions[
                model_name
            ][
                test_idx[
                    0
                ]
            ] = pred


            fold_predictions[
                model_name
            ] = pred


            cleaned_params = (
                clean_best_params(

                    model_name,
                    search.best_params_
                )
            )


            parameter_history[
                model_name
            ].append(
                {

                    "Outer_fold":
                        outer_fold,

                    "Held_out_condition":
                        held_out_id,

                    "Inner_RMSE":
                        -float(
                            search.best_score_
                        ),

                    **cleaned_params
                }
            )


        # ----------------------------------------------------
        # Store outer-fold predictions
        # ----------------------------------------------------

        prediction_rows.append(

            {

                "Outer_fold":
                    outer_fold,

                "Condition_ID":
                    held_out_id,

                "FPU":
                    float(
                        X_test[
                            0,
                            0
                        ]
                    ),

                "Substrate":
                    float(
                        X_test[
                            0,
                            1
                        ]
                    ),

                "Ratio":
                    float(
                        X_test[
                            0,
                            2
                        ]
                    ),

                "Observed":
                    float(
                        y_test[
                            0
                        ]
                    ),

                "Quadratic_pred":
                    fold_predictions[
                        "Quadratic"
                    ],

                "SVR_pred":
                    fold_predictions[
                        "SVR"
                    ],

                "GPR_pred":
                    fold_predictions[
                        "GPR"
                    ],

                "SplineRidge_pred":
                    fold_predictions[
                        "SplineRidge"
                    ]
            }
        )


        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (

            outer_fold == 1

            or

            outer_fold % 5 == 0

            or

            outer_fold == n_samples
        ):

            elapsed = (
                time.time()
                -
                start_time
            )


            print(
                f"Outer fold "
                f"{outer_fold:>2}/"
                f"{n_samples}"
                f" | held out = "
                f"{held_out_id}"
                f" | elapsed = "
                f"{elapsed:.1f} s"
            )


    # ========================================================
    # 8. FINAL OUTER PERFORMANCE
    # ========================================================

    summary_rows = []


    for model_name in MODEL_LIST:


        pred = predictions[
            model_name
        ]


        if np.isnan(
            pred
        ).any():

            raise RuntimeError(
                f"Incomplete outer predictions: "
                f"{model_name}"
            )


        metrics = calculate_metrics(

            y_true=
                y,

            y_pred=
                pred
        )


        summary_rows.append(

            {

                "Condition":
                    condition_name,

                "Model":
                    model_name,

                "n":
                    n_samples,

                "R2":
                    metrics[
                        "R2"
                    ],

                "RMSE":
                    metrics[
                        "RMSE"
                    ],

                "MAE":
                    metrics[
                        "MAE"
                    ],

                "MRE":
                    metrics[
                        "MRE"
                    ]
            }
        )


    summary = pd.DataFrame(
        summary_rows
    )


    summary = (

        summary

        .sort_values(
            by="RMSE",
            ascending=True
        )

        .reset_index(
            drop=True
        )
    )


    summary[
        "RMSE_rank"
    ] = (

        np.arange(
            1,
            len(summary) + 1
        )
    )


    predictions_df = pd.DataFrame(
        prediction_rows
    )


    params_df = {

        model_name:
            pd.DataFrame(
                parameter_history[
                    model_name
                ]
            )

        for model_name in MODEL_LIST
    }


    print("\n")
    print("=" * 120)

    print(
        "FINAL OUTER-CV PERFORMANCE — "
        f"{condition_name.upper()}"
    )

    print("=" * 120)


    print(

        summary

        .round(
            {
                "R2": 4,
                "RMSE": 4,
                "MAE": 4,
                "MRE": 4
            }
        )

        .to_string(
            index=False
        )
    )


    print("\n")
    print(
        "All reported metrics above were calculated "
        "ONLY from outer held-out predictions."
    )


    return {

        "summary":
            summary,

        "predictions":
            predictions_df,

        "parameter_history":
            params_df,

        "raw_predictions":
            predictions
    }


# ============================================================
# 9. RUN WITHOUT BARK
# ============================================================

nested_without = (
    run_fully_nested_condition_loocv(

        condition_without,

        "without bark"
    )
)


# ============================================================
# 10. RUN WITH BARK
# ============================================================

nested_with = (
    run_fully_nested_condition_loocv(

        condition_with,

        "with bark"
    )
)


# ============================================================
# 11. COMBINED TABLE 4 CANDIDATE
# ============================================================

table4_revised = pd.concat(

    [

        nested_without[
            "summary"
        ],

        nested_with[
            "summary"
        ]
    ],

    ignore_index=True
)


print("\n")
print("=" * 120)

print(
    "REVISED TABLE 4 — "
    "FULLY NESTED CONDITION-LEVEL PERFORMANCE"
)

print("=" * 120)


print(

    table4_revised

    .round(
        {
            "R2": 3,
            "RMSE": 3,
            "MAE": 3,
            "MRE": 3
        }
    )

    .to_string(
        index=False
    )
)


# ============================================================
# 12. PARAMETER-SELECTION FREQUENCY
# ============================================================

def print_parameter_frequency(
    nested_result,
    condition_name
):

    print("\n")
    print("=" * 120)

    print(
        "HYPERPARAMETER SELECTION FREQUENCY — "
        f"{condition_name.upper()}"
    )

    print("=" * 120)


    for model_name in MODEL_LIST:

        df = nested_result[
            "parameter_history"
        ][
            model_name
        ]


        parameter_columns = [

            col

            for col in df.columns

            if col not in [

                "Outer_fold",
                "Held_out_condition",
                "Inner_RMSE"
            ]
        ]


        print(
            f"\n[{model_name}]"
        )


        if len(
            parameter_columns
        ) == 0:

            continue


        combo_counts = (

            df[
                parameter_columns
            ]

            .astype(str)

            .value_counts()

            .reset_index(
                name="Selection_count"
            )
        )


        print(

            combo_counts

            .head(
                10
            )

            .to_string(
                index=False
            )
        )


print_parameter_frequency(

    nested_without,

    "without bark"
)


print_parameter_frequency(

    nested_with,

    "with bark"
)


# ============================================================
# 13. FINAL AUDIT
# ============================================================

print("\n")
print("=" * 120)
print("REVIEWER 3 NESTED-CV AUDIT")
print("=" * 120)

print(
    "Without bark outer predictions:",
    len(
        nested_without[
            "predictions"
        ]
    ),
    "/ expected 23"
)

print(
    "With bark outer predictions   :",
    len(
        nested_with[
            "predictions"
        ]
    ),
    "/ expected 29"
)

print(
    "Outer held-out factor combinations "
    "were absent from their corresponding "
    "training sets."
)

print(
    "All tuning and preprocessing were "
    "performed exclusively within outer-training data."
)

print(
    "Final metrics were calculated from "
    "pooled outer predictions only."
)

print("=" * 120)
print("STATUS: FULLY NESTED CONDITION-LEVEL CV COMPLETE")
print("=" * 120)


In [ ]:
import os

# ============================================================
# PREPARE FINAL OUTER-FOLD PREDICTIONS FOR FIGS. 3 AND 4
# ============================================================

pred_without = nested_without["raw_predictions"]
pred_with = nested_with["raw_predictions"]

# Save final outer-fold prediction tables for auditing.
os.makedirs("results", exist_ok=True)

nested_without["predictions"].to_csv(
    "results/outer_predictions_without_bark.csv",
    index=False
)

nested_with["predictions"].to_csv(
    "results/outer_predictions_with_bark.csv",
    index=False
)

table4_revised.to_csv(
    "results/table4_recomputed_metrics.csv",
    index=False
)

print("Saved final outer-fold prediction tables and recomputed metrics.")


In [ ]:
# ============================================================
# FINAL FIGURES 3 & 4 — 10 SEPARATE PLOTS
#
# STYLE MATCHED TO THE ORIGINAL MOTIF CODE
# ------------------------------------------------------------
# No panel letters
# No plot titles
# No model names inside figures
# No R2 / RMSE text inside figures
#
# HEATMAP
#   Figure size : 5.5 × 5.0
#   Colormap    : RdBu_r
#   Labels      : 13 pt bold
#   Coefficients: 13 pt bold
#   Colorbar    : 11 pt
#
# SCATTER
#   Figure size : 5.0 × 4.5
#   Axis labels : 14 pt bold
#   Tick labels : 11 pt
#   Marker      : white circle + black edge
#   Marker size : 72
#   1:1 line    : red
#   Major + minor grid
#   Full box frame
#
# IMPORTANT
# ------------------------------------------------------------
# Uses existing PRIMARY nested-LOOCV predictions.
# NO model refitting.
#
# Required objects:
#   condition_without
#   condition_with
#   pred_without
#   pred_with
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib import rcParams
from matplotlib.ticker import AutoMinorLocator

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)


# ============================================================
# 1. CHECK REQUIRED OBJECTS
# ============================================================

required_objects = [
    "condition_without",
    "condition_with",
    "pred_without",
    "pred_with"
]


for obj_name in required_objects:

    if obj_name not in globals():

        raise RuntimeError(
            f"{obj_name} not found. "
            "Run the primary condition-level nested-LOOCV "
            "and prediction-extraction cells first."
        )


# ============================================================
# 2. DATA CHECK
# ============================================================

assert len(condition_without) == 23
assert len(condition_with) == 29


assert (
    condition_without
    .duplicated(
        subset=[
            "FPU",
            "Substrate",
            "Ratio"
        ]
    )
    .sum()
    == 0
)


assert (
    condition_with
    .duplicated(
        subset=[
            "FPU",
            "Substrate",
            "Ratio"
        ]
    )
    .sum()
    == 0
)


print("=" * 100)
print("FINAL FIGURE DATA CHECK")
print("=" * 100)

print(
    f"Without bark : "
    f"{len(condition_without)} unique conditions"
)

print(
    f"With bark    : "
    f"{len(condition_with)} unique conditions"
)


# ============================================================
# 3. GLOBAL FIGURE STYLE
#
# MATCHED TO ORIGINAL MOTIF
# ============================================================

rcParams["font.family"] = "Arial"

rcParams["axes.linewidth"] = 1.0

rcParams["xtick.direction"] = "out"
rcParams["ytick.direction"] = "out"

rcParams["xtick.major.width"] = 1.0
rcParams["ytick.major.width"] = 1.0

rcParams["xtick.major.size"] = 4
rcParams["ytick.major.size"] = 4

rcParams["xtick.minor.width"] = 0.8
rcParams["ytick.minor.width"] = 0.8

rcParams["xtick.minor.size"] = 2.5
rcParams["ytick.minor.size"] = 2.5

rcParams["xtick.labelsize"] = 11
rcParams["ytick.labelsize"] = 11

rcParams["pdf.fonttype"] = 42
rcParams["ps.fonttype"] = 42


# ============================================================
# 4. SAVE OPTION
# ============================================================

SAVE_FIG = True

DPI = 600

SAVE_DIR = "figures/Figure3_4"


if SAVE_FIG:

    os.makedirs(
        SAVE_DIR,
        exist_ok=True
    )


# ============================================================
# 5. MODEL ORDER
# ============================================================

MODEL_ORDER = [
    "Quadratic",
    "SVR",
    "GPR",
    "SplineRidge"
]


# ============================================================
# 6. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )


    r2 = r2_score(
        y_true,
        y_pred
    )


    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


    mae = mean_absolute_error(
        y_true,
        y_pred
    )


    mre = np.mean(

        np.abs(
            y_true - y_pred
        )

        /

        np.maximum(
            np.abs(y_true),
            1e-12
        )
    )


    return {
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MRE": mre
    }


# ============================================================
# 7. PRIMARY METRIC CHECK
# ============================================================

print("\n")
print("=" * 100)
print("PRIMARY ANALYSIS METRIC CHECK")
print("=" * 100)


for condition_name, data, predictions in [

    (
        "WITHOUT BARK",
        condition_without,
        pred_without
    ),

    (
        "WITH BARK",
        condition_with,
        pred_with
    )
]:

    print(
        f"\n{condition_name}"
    )


    y_true = (

        data[
            "Saccharification"
        ]

        .to_numpy(
            dtype=float
        )
    )


    for model in MODEL_ORDER:

        metrics = calculate_metrics(

            y_true,

            predictions[
                model
            ]
        )


        print(

            f"{model:<13s} | "
            f"R2 = {metrics['R2']:.4f} | "
            f"RMSE = {metrics['RMSE']:.4f} | "
            f"MAE = {metrics['MAE']:.4f} | "
            f"MRE = {metrics['MRE']:.4f}"
        )


# ============================================================
# 8. SAVE FUNCTION
# ============================================================

def save_plot(
    fig,
    basename
):


    if not SAVE_FIG:

        return


    fig.savefig(

        os.path.join(
            SAVE_DIR,
            basename + ".png"
        ),

        dpi=DPI,

        bbox_inches="tight",

        pad_inches=0.05
    )


    fig.savefig(

        os.path.join(
            SAVE_DIR,
            basename + ".tiff"
        ),

        dpi=DPI,

        bbox_inches="tight",

        pad_inches=0.05
    )


    fig.savefig(

        os.path.join(
            SAVE_DIR,
            basename + ".pdf"
        ),

        bbox_inches="tight",

        pad_inches=0.05
    )


# ============================================================
# 9. CORRELATION HEATMAP
#
# MATCHED TO MOTIF SIZE/TEXT
# CURRENT RdBu_r COLOR RETAINED
# ============================================================

def plot_heatmap(
    data,
    basename
):


    # --------------------------------------------------------
    # Correlation data
    # --------------------------------------------------------

    corr_data = pd.DataFrame({

        "FPU":
            data[
                "FPU"
            ].to_numpy(
                dtype=float
            ),

        "Substrate":
            data[
                "Substrate"
            ].to_numpy(
                dtype=float
            ),

        "KMF006":
            data[
                "Ratio"
            ].to_numpy(
                dtype=float
            ),

        "SR":
            data[
                "Saccharification"
            ].to_numpy(
                dtype=float
            )
    })


    corr_df = corr_data.corr(
        method="pearson"
    )


    # --------------------------------------------------------
    # Figure size EXACTLY matched to motif
    # --------------------------------------------------------

    fig, ax = plt.subplots(

        figsize=(
            5.5,
            5.0
        )
    )


    fig.patch.set_facecolor(
        "white"
    )


    ax.set_facecolor(
        "white"
    )


    # --------------------------------------------------------
    # Heatmap
    #
    # KEEP CURRENT RdBu_r
    # --------------------------------------------------------

    im = ax.imshow(

        corr_df.values,

        cmap="RdBu_r",

        vmin=-1,
        vmax=1,

        aspect="equal"
    )


    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    labels = [
        "FPU",
        "Substrate",
        "KMF006",
        "SR"
    ]


    ax.set_xticks(
        np.arange(
            len(labels)
        )
    )


    ax.set_yticks(
        np.arange(
            len(labels)
        )
    )


    ax.set_xticklabels(

        labels,

        fontsize=13,

        fontweight="bold"
    )


    ax.set_yticklabels(

        labels,

        fontsize=13,

        fontweight="bold"
    )


    # --------------------------------------------------------
    # Correlation coefficients
    # --------------------------------------------------------

    for i in range(
        corr_df.shape[0]
    ):

        for j in range(
            corr_df.shape[1]
        ):

            value = corr_df.iloc[
                i,
                j
            ]


            text_color = (

                "white"

                if abs(
                    value
                ) >= 0.45

                else "black"
            )


            ax.text(

                j,
                i,

                f"{value:.2f}",

                ha="center",
                va="center",

                fontsize=13,

                fontweight="bold",

                color=text_color
            )


    # --------------------------------------------------------
    # Colorbar
    # --------------------------------------------------------

    cbar = fig.colorbar(

        im,

        ax=ax,

        fraction=0.046,

        pad=0.04
    )


    cbar.ax.tick_params(

        labelsize=11,

        width=1.0,

        length=4
    )


    # --------------------------------------------------------
    # Frame
    # --------------------------------------------------------

    for spine in ax.spines.values():

        spine.set_visible(
            True
        )

        spine.set_linewidth(
            1.0
        )


    ax.tick_params(
        length=0
    )


    # --------------------------------------------------------
    # NO TITLE
    # NO PANEL LETTER
    # --------------------------------------------------------

    plt.tight_layout()


    save_plot(
        fig,
        basename
    )


    plt.show()


# ============================================================
# 10. COMMON AXIS LIMITS
#
# Same scale across the four models within each bark dataset.
# ============================================================

def get_dataset_axis_limits(
    data,
    predictions,
    padding=2.0
):


    measured = (

        data[
            "Saccharification"
        ]

        .to_numpy(
            dtype=float
        )
    )


    values = [
        measured
    ]


    for model in MODEL_ORDER:

        values.append(

            np.asarray(
                predictions[
                    model
                ],
                dtype=float
            )
        )


    values = np.concatenate(
        values
    )


    min_value = np.min(
        values
    )


    max_value = np.max(
        values
    )


    lower = np.floor(

        (
            min_value
            -
            padding
        )

        / 5

    ) * 5


    upper = np.ceil(

        (
            max_value
            +
            padding
        )

        / 5

    ) * 5


    return (
        lower,
        upper
    )


# ============================================================
# 11. SCATTER PLOT
#
# MOTIF SIZE / FONT / GRID / FRAME
# CURRENT CIRCLE POINT DESIGN
# POINT SIZE = 72
# ============================================================

def plot_scatter(
    data,
    predicted,
    axis_min,
    axis_max,
    basename
):


    measured = (

        data[
            "Saccharification"
        ]

        .to_numpy(
            dtype=float
        )
    )


    predicted = np.asarray(
        predicted,
        dtype=float
    )


    # --------------------------------------------------------
    # Figure size EXACTLY matched to motif
    # --------------------------------------------------------

    fig, ax = plt.subplots(

        figsize=(
            5.0,
            4.5
        )
    )


    # --------------------------------------------------------
    # PURE WHITE BACKGROUND
    # --------------------------------------------------------

    fig.patch.set_facecolor(
        "white"
    )


    ax.set_facecolor(
        "white"
    )


    # --------------------------------------------------------
    # DATA POINTS
    #
    # CURRENT DESIGN RETAINED
    # SIZE INCREASED TO 72
    # --------------------------------------------------------

    ax.scatter(

        measured,
        predicted,

        s=72,

        marker="o",

        facecolors="white",

        edgecolors="black",

        linewidths=1.1,

        zorder=5
    )


    # --------------------------------------------------------
    # RED 1:1 REFERENCE LINE
    #
    # MATCHED TO MOTIF
    # --------------------------------------------------------

    ax.plot(

        [
            axis_min,
            axis_max
        ],

        [
            axis_min,
            axis_max
        ],

        color="red",

        linewidth=1.3,

        zorder=3
    )


    # --------------------------------------------------------
    # X/Y RANGE
    # --------------------------------------------------------

    ax.set_xlim(
        axis_min,
        axis_max
    )


    ax.set_ylim(
        axis_min,
        axis_max
    )


    # --------------------------------------------------------
    # AXIS TITLES
    #
    # 14 pt BOLD — MATCHED TO MOTIF
    # --------------------------------------------------------

    ax.set_xlabel(

        "Measured saccharification yield (%)",

        fontsize=14,

        fontweight="bold",

        labelpad=8
    )


    ax.set_ylabel(

        "Predicted saccharification yield (%)",

        fontsize=14,

        fontweight="bold",

        labelpad=8
    )


    # --------------------------------------------------------
    # MAJOR TICKS
    #
    # 11 pt — MATCHED TO MOTIF
    # --------------------------------------------------------

    ax.tick_params(

        axis="both",

        which="major",

        labelsize=11,

        width=1.0,

        length=4,

        direction="out"
    )


    # --------------------------------------------------------
    # MINOR TICKS
    # --------------------------------------------------------

    ax.xaxis.set_minor_locator(
        AutoMinorLocator()
    )


    ax.yaxis.set_minor_locator(
        AutoMinorLocator()
    )


    ax.tick_params(

        axis="both",

        which="minor",

        width=0.8,

        length=2.5,

        direction="out"
    )


    # --------------------------------------------------------
    # MAJOR GRID
    #
    # MATCHED TO MOTIF
    # --------------------------------------------------------

    ax.grid(

        True,

        which="major",

        linestyle=(
            0,
            (
                3,
                3
            )
        ),

        linewidth=0.75,

        color="#969696",

        alpha=0.85,

        zorder=0
    )


    # --------------------------------------------------------
    # MINOR GRID
    #
    # MATCHED TO MOTIF
    # --------------------------------------------------------

    ax.grid(

        True,

        which="minor",

        linestyle=(
            0,
            (
                1,
                3
            )
        ),

        linewidth=0.45,

        color="#C8C8C8",

        alpha=0.90,

        zorder=0
    )


    # --------------------------------------------------------
    # FULL BOX FRAME
    #
    # MATCHED TO MOTIF
    # --------------------------------------------------------

    for spine in [

        "left",
        "right",
        "top",
        "bottom"

    ]:

        ax.spines[
            spine
        ].set_visible(
            True
        )


        ax.spines[
            spine
        ].set_linewidth(
            1.0
        )


    # --------------------------------------------------------
    # NO TITLE
    # NO MODEL NAME
    # NO PANEL LETTER
    # NO R2
    # NO RMSE
    # --------------------------------------------------------

    plt.tight_layout()


    save_plot(
        fig,
        basename
    )


    plt.show()


# ============================================================
# 12. AXIS LIMITS
# ============================================================

without_min, without_max = (
    get_dataset_axis_limits(
        condition_without,
        pred_without
    )
)


with_min, with_max = (
    get_dataset_axis_limits(
        condition_with,
        pred_with
    )
)


print("\n")
print("=" * 100)
print("SCATTER AXIS LIMITS")
print("=" * 100)

print(
    f"Without bark : "
    f"{without_min:.0f} to {without_max:.0f}"
)

print(
    f"With bark    : "
    f"{with_min:.0f} to {with_max:.0f}"
)


# ============================================================
# 13. FIGURE 3 — WITHOUT BARK
# ============================================================

print("\n")
print("=" * 100)
print("FIGURE 3 — WITHOUT BARK")
print("=" * 100)


# Correlation
plot_heatmap(

    data=
        condition_without,

    basename=
        "Fig3_without_bark_correlation"
)


# Quadratic
plot_scatter(

    data=
        condition_without,

    predicted=
        pred_without[
            "Quadratic"
        ],

    axis_min=
        without_min,

    axis_max=
        without_max,

    basename=
        "Fig3_without_bark_quadratic"
)


# SVR
plot_scatter(

    data=
        condition_without,

    predicted=
        pred_without[
            "SVR"
        ],

    axis_min=
        without_min,

    axis_max=
        without_max,

    basename=
        "Fig3_without_bark_SVR"
)


# GPR
plot_scatter(

    data=
        condition_without,

    predicted=
        pred_without[
            "GPR"
        ],

    axis_min=
        without_min,

    axis_max=
        without_max,

    basename=
        "Fig3_without_bark_GPR"
)


# SplineRidge
plot_scatter(

    data=
        condition_without,

    predicted=
        pred_without[
            "SplineRidge"
        ],

    axis_min=
        without_min,

    axis_max=
        without_max,

    basename=
        "Fig3_without_bark_SplineRidge"
)


# ============================================================
# 14. FIGURE 4 — WITH BARK
# ============================================================

print("\n")
print("=" * 100)
print("FIGURE 4 — WITH BARK")
print("=" * 100)


# Correlation
plot_heatmap(

    data=
        condition_with,

    basename=
        "Fig4_with_bark_correlation"
)


# Quadratic
plot_scatter(

    data=
        condition_with,

    predicted=
        pred_with[
            "Quadratic"
        ],

    axis_min=
        with_min,

    axis_max=
        with_max,

    basename=
        "Fig4_with_bark_quadratic"
)


# SVR
plot_scatter(

    data=
        condition_with,

    predicted=
        pred_with[
            "SVR"
        ],

    axis_min=
        with_min,

    axis_max=
        with_max,

    basename=
        "Fig4_with_bark_SVR"
)


# GPR
plot_scatter(

    data=
        condition_with,

    predicted=
        pred_with[
            "GPR"
        ],

    axis_min=
        with_min,

    axis_max=
        with_max,

    basename=
        "Fig4_with_bark_GPR"
)


# SplineRidge
plot_scatter(

    data=
        condition_with,

    predicted=
        pred_with[
            "SplineRidge"
        ],

    axis_min=
        with_min,

    axis_max=
        with_max,

    basename=
        "Fig4_with_bark_SplineRidge"
)


# ============================================================
# 15. FINAL OUTPUT ORDER
# ============================================================

print("\n")
print("=" * 100)
print("FINAL OUTPUT ORDER")
print("=" * 100)

print(
    """
1  Fig3_without_bark_correlation
2  Fig3_without_bark_quadratic
3  Fig3_without_bark_SVR
4  Fig3_without_bark_GPR
5  Fig3_without_bark_SplineRidge

6  Fig4_with_bark_correlation
7  Fig4_with_bark_quadratic
8  Fig4_with_bark_SVR
9  Fig4_with_bark_GPR
10 Fig4_with_bark_SplineRidge
"""
)

print("=" * 100)
print("ALL FINAL PANELS COMPLETE")
print("=" * 100)
